# FINAL 00 — Configuration, Frozen Rules and Run Manifest

**Thesis:** An Ethical Risk Assessment of Sentiment Analysis Models in Social Media Monitoring with a Focus on Bias, Inclusivity, and Misclassification Patterns  
**Student:** Abhishek Tomar | PN1196973 | LJMU | October 2026

**Purpose:** Establish one authoritative configuration before any final experiment is run.

> **Final-run rule:** TweetEval test data is held out from model/feature/hyperparameter selection. External datasets are never used for source-model selection. Linguistic subgroup thresholds are frozen and are not tuned after observing subgroup sizes.

## What this notebook freezes
- dataset roles and paths;
- TweetEval official split counts;
- linguistic subgroup thresholds and terminology;
- minimum subgroup size for stable quantitative interpretation;
- random seed;
- explicit prohibition on test-driven model/feature/hyperparameter selection;
- output folders and run manifest.

In [1]:
!pip install -q emoji vaderSentiment pandas pyarrow scikit-learn scipy google-api-python-client

from google.colab import drive, auth
drive.mount('/content/drive')

import io
import sys
import json
import platform
from pathlib import Path

import google.auth
import numpy as np
import pandas as pd
import sklearn
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# ------------------------------------------------------------------
# Load the exact shared utility file by Drive file ID.
# This avoids assuming that the shared "Experiments" folder is mounted
# as /content/drive/MyDrive/Experiments.
# ------------------------------------------------------------------
UTIL_FILE_ID = "14KfwtUf2zSd_jlW7i-6K8XgBJ62Aqk02"
UTIL_LOCAL = Path("/content/final_thesis_utils.py")

auth.authenticate_user()
creds, _ = google.auth.default()
service = build("drive", "v3", credentials=creds, cache_discovery=False)

request = service.files().get_media(fileId=UTIL_FILE_ID)
with io.FileIO(str(UTIL_LOCAL), "wb") as fh:
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        status, done = downloader.next_chunk()
        if status:
            print(f"Downloading final_thesis_utils.py: {int(status.progress()*100)}%")

assert UTIL_LOCAL.exists() and UTIL_LOCAL.stat().st_size > 0, \
    "final_thesis_utils.py could not be downloaded."

sys.path.insert(0, "/content")
import final_thesis_utils as ftu

# ------------------------------------------------------------------
# Stable execution-output location.
# The notebooks themselves remain in the shared Experiments folder.
# Generated final-run artefacts go here in the user's mounted MyDrive.
# ------------------------------------------------------------------
RUN_ROOT = Path("/content/drive/MyDrive/LJMU_FINAL_EXPERIMENT_RUN")

ftu.FINAL_ROOT = RUN_ROOT
ftu.FINAL = {
    "root": RUN_ROOT,
    "data": RUN_ROOT / "data",
    "models": RUN_ROOT / "models",
    "predictions": RUN_ROOT / "predictions",
    "results": RUN_ROOT / "results",
    "figures": RUN_ROOT / "figures",
    "manifests": RUN_ROOT / "manifests",
}

# Export the shared names only after overriding the module configuration.
from final_thesis_utils import *

FINAL_ROOT = RUN_ROOT
FINAL = ftu.FINAL

setup_final_dirs()

print("\n✅ final_thesis_utils imported successfully")
print("✅ Utility source Drive ID:", UTIL_FILE_ID)
print("✅ Final run root:", FINAL_ROOT)
print("\nFrozen subgroup configuration:")
print(json.dumps(FREEZE_CONFIG, indent=2))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 4.5 MB/s eta 0:00:00
Mounted at /content/drive



✅ final_thesis_utils imported successfully
✅ Utility source Drive ID: 14KfwtUf2zSd_jlW7i-6K8XgBJ62Aqk02
✅ Final run root: /content/drive/MyDrive/LJMU_FINAL_EXPERIMENT_RUN

Frozen subgroup configuration:
{
  "emoji_heavy": {
    "formula": "emoji_count / word_count",
    "operator": ">",
    "threshold": 0.05
  },
  "slang_heavy": {
    "formula": "slang_count / word_count",
    "operator": ">",
    "threshold": 0.1,
    "lexicon": [
      "lol",
      "lmao",
      "omg",
      "tbh",
      "imo",
      "imho",
      "wtf",
      "idk",
      "ngl",
      "smh",
      "af",
      "fr",
      "lowkey",
      "highkey",
      "slay",
      "lit",
      "goat",
      "vibe",
      "fire",
      "wanna",
      "gonna",
      "gotta",
      "kinda",
      "ur",
      "bruh",
      "bro",
      "fam",
      "dope",
      "sick",
      "sus",
      "bussin",
      "salty",
      "cap",
      "bet",
      "flex",
      "ghost",
      "clout",
      "woke",
      "ftw",
      "ftl",
      "epi

In [2]:
# Verify that the source prepared data exists.
required = [
    LEGACY["data"]/"tw_train.parquet",
    LEGACY["data"]/"tw_val.parquet",
    LEGACY["data"]/"tw_test.parquet",
    LEGACY["data"]/"se_test.parquet",
    LEGACY["data"]/"s140_test.parquet",
]
missing=[str(p) for p in required if not p.exists()]
assert not missing, f"Missing required source files: {missing}"
print("All source files found.")

All source files found.


In [3]:
manifest = {
    "run_name":"LJMU_FINAL_LOCKED_RUN_2026",
    "seed":SEED,
    "tweet_eval_role":"primary model development and held-out final evaluation",
    "sentiment140_role":"binary external robustness condition",
    "semeval_role":"domain/task-shift stress condition",
    "sarcoji_role":"auxiliary proxy-validation/limitation evidence only",
    "selection_policy":"train/validation/CV only; TweetEval test untouched until configuration lock",
    "external_selection_policy":"external datasets never used for source model or feature selection",
    "subgroup_freeze":FREEZE_CONFIG,
    "python":platform.python_version(),
    "sklearn":sklearn.__version__,
}
save_json(manifest, FINAL["manifests"]/"FINAL_RUN_MANIFEST.json")
save_json(FREEZE_CONFIG, FINAL["manifests"]/"SUBGROUP_FREEZE_CONFIG.json")
print(json.dumps(manifest,indent=2))

{
  "run_name": "LJMU_FINAL_LOCKED_RUN_2026",
  "seed": 42,
  "tweet_eval_role": "primary model development and held-out final evaluation",
  "sentiment140_role": "binary external robustness condition",
  "semeval_role": "domain/task-shift stress condition",
  "sarcoji_role": "auxiliary proxy-validation/limitation evidence only",
  "selection_policy": "train/validation/CV only; TweetEval test untouched until configuration lock",
  "external_selection_policy": "external datasets never used for source model or feature selection",
  "subgroup_freeze": {
    "emoji_heavy": {
      "formula": "emoji_count / word_count",
      "operator": ">",
      "threshold": 0.05
    },
    "slang_heavy": {
      "formula": "slang_count / word_count",
      "operator": ">",
      "threshold": 0.1,
      "lexicon": [
        "lol",
        "lmao",
        "omg",
        "tbh",
        "imo",
        "imho",
        "wtf",
        "idk",
        "ngl",
        "smh",
        "af",
        "fr",
        "lo